In [ ]:
# Camada Silver 

import sqlite3
from datetime import date
import pandas as pd


In [2]:
conexao  = sqlite3.connect(database='db_project_eng_dados')

In [ ]:
# Forma CORRETA de criar a camada Silver com tipagem de dados definidas no SQLite
# 1º passo é criar a tabela silver definindo os tipos de dados corretos para essa camada.
# O SQLite não permite alterar os tipos de dados da tabela 'schema real', comum em SGBDs.

conexao.execute("""
CREATE TABLE IF NOT EXISTS silver_produtos (
    id_produto INTEGER,
    nome_produto TEXT,
    categoria TEXT,
    tipo_dado TEXT,
    resolucao_espacial REAL,
    sistema_coordenadas TEXT,
    area_cobertura_km2 REAL,
    data_aquisicao DATE,
    formato TEXT,
    fornecedor TEXT,
    preco REAL,
    data_bronze DATE,
    data_silver DATE );""")

# 2º Inserir os dados da tabela bronze
# Inserção com conversões
conexao.execute("""
INSERT INTO silver_produtos
SELECT DISTINCT 
    CAST(id_produto AS INTEGER),
    nome_produto,
    categoria,
    tipo_dado,
    CAST (resolucao_espacial AS REAL),
    sistema_coordenadas,
    CAST (area_cobertura_km2 AS REAL),
    date (data_aquisicao),
    formato,
    fornecedor,
    CAST (preco AS REAL),
    date(data_bronze),
    date('now')
FROM bronze_produtos;                                                                                                                                                                                                 
                                   """)
  

conexao.commit()

# Sem duplicação com DISTINCT
# Padronização de datas com nova coluna de linhagem (data_silver)
# Definição dos tipos de dados 


In [ ]:
# Caso tenha algum problema de duplicação de dados por parte de sintaxe ou do módulo:

conexao.execute("DROP TABLE IF EXISTS silver_produtos;")
conexao.commit()

In [ ]:
# Validação de tipagem das colunas na camada Silver:


pd.read_sql("""
SELECT
    id_produto, 
    typeof(id_produto) AS tipo_id_produto,
    resolucao_espacial,
    typeof(resolucao_espacial) AS tipo_resolucao,
    area_cobertura_km2,
    typeof(area_cobertura_km2) AS tipo_area,
    preco,
    typeof(preco) AS tipo_preco,
    data_aquisicao,
    typeof (data_aquisicao) AS tipo_data_aquisicao
FROM silver_produtos
LIMIT 5;

""", conexao)

,id_produto,tipo_id_produto,resolucao_espacial,tipo_resolucao,area_cobertura_km2,tipo_area,preco,tipo_preco,data_aquisicao,tipo_data_aquisicao
0,1,integer,10.0,real,1500.0,real,3500.0,real,2023-06-15,text
1,2,integer,30.0,real,5000.0,real,0.0,real,2022-11-20,text
2,3,integer,0.5,real,800.0,real,12000.0,real,2023-02-10,text
3,4,integer,1.0,real,3200.0,real,0.0,real,2021-08-05,text
4,5,integer,20.0,real,2100.0,real,2500.0,real,2022-09-30,text


In [43]:
# Visualização amostral da Camada Silver:

df_silver_check = pd.read_sql ("""
SELECT * FROM silver_produtos
LIMIT 3 """, conexao)

df_silver_check

,id_produto,nome_produto,categoria,tipo_dado,resolucao_espacial,sistema_coordenadas,area_cobertura_km2,data_aquisicao,formato,fornecedor,preco,data_bronze,data_silver
0,1,Mapa de Uso do Solo 2023,Mapeamento Temático,Vetorial,10.0,SIRGAS 2000 / UTM 23S,1500.0,2023-06-15,Shapefile,GeoMapas Ltda,3500.0,2025-12-22,2025-12-23
1,2,Modelo Digital de Elevação,MDE,Raster,30.0,WGS84,5000.0,2022-11-20,GeoTIFF,INPE,0.0,2025-12-22,2025-12-23
2,3,Ortoimagem Urbana São Paulo,Imagem Orbital,Raster,0.5,SIRGAS 2000,800.0,2023-02-10,GeoTIFF,Maxar,12000.0,2025-12-22,2025-12-23


In [44]:
# Métricas de qualidade:

df_metricas_silver = pd.read_sql ("""
SELECT 
    COUNT (*) AS total_registros,
    COUNT (DISTINCT id_produto) AS produtos_unicos,
    SUM (preco IS NULL) AS preco_nulo,
    MIN (data_silver) AS data_processamento
FROM silver_produtos """, conexao)

df_metricas_silver




,total_registros,produtos_unicos,preco_nulo,data_processamento
0,7,7,0,2025-12-23


In [45]:
# Visualização total da tabela SILVER:

df_table_silver = pd.read_sql ("""
SELECT * FROM silver_produtos """, conexao)

df_table_silver

,id_produto,nome_produto,categoria,tipo_dado,resolucao_espacial,sistema_coordenadas,area_cobertura_km2,data_aquisicao,formato,fornecedor,preco,data_bronze,data_silver
0,1,Mapa de Uso do Solo 2023,Mapeamento Temático,Vetorial,10.0,SIRGAS 2000 / UTM 23S,1500.0,2023-06-15,Shapefile,GeoMapas Ltda,3500.0,2025-12-22,2025-12-23
1,2,Modelo Digital de Elevação,MDE,Raster,30.0,WGS84,5000.0,2022-11-20,GeoTIFF,INPE,0.0,2025-12-22,2025-12-23
2,3,Ortoimagem Urbana São Paulo,Imagem Orbital,Raster,0.5,SIRGAS 2000,800.0,2023-02-10,GeoTIFF,Maxar,12000.0,2025-12-22,2025-12-23
3,4,Mapa de Drenagem Hidrográfica,Hidrografia,Vetorial,1.0,SIRGAS 2000,3200.0,2021-08-05,GeoPackage,ANA,0.0,2025-12-22,2025-12-23
4,5,Classificação de Vegetação Cerrado,Vegetação,Raster,20.0,WGS84,2100.0,2022-09-30,GeoTIFF,IBGE,2500.0,2025-12-22,2025-12-23
5,6,Limites Administrativos Municipais,Base Cartográfica,Vetorial,1.0,SIRGAS 2000,8500.0,2023-01-01,Shapefile,IBGE,0.0,2025-12-22,2025-12-23
6,7,Mapa de Risco de Deslizamento,Análise Ambiental,Vetorial,1.0,SIRGAS 2000 / UTM 22S,600.0,2023-07-12,GeoPackage,Defesa Civil,4800.0,2025-12-22,2025-12-23


In [ ]:
# O pandas considera strings como objetos no dataframe:

df_table_silver.dtypes



id_produto              object
nome_produto            object
categoria               object
tipo_dado               object
resolucao_espacial      object
sistema_coordenadas     object
area_cobertura_km2     float64
data_aquisicao          object
formato                 object
fornecedor              object
preco                  float64
data_bronze             object
data_silver             object
dtype: object

In [ ]:
# Alteração de tipagem de dados no pandas:

df_table_silver_2 = df_table_silver
df_table_silver_2 = df_table_silver_2.astype(
    {'id_produto' : int,
     'area_cobertura_km2':float,
     }
)

df_table_silver_2.dtypes

# Foi criada uma variável cópia para a tabela com os tipos de dados.


id_produto               int64
nome_produto            object
categoria               object
tipo_dado               object
resolucao_espacial      object
sistema_coordenadas     object
area_cobertura_km2     float64
data_aquisicao          object
formato                 object
fornecedor              object
preco                  float64
data_bronze             object
data_silver             object
dtype: object

In [ ]:
# Visualização de tipagem de tabela SILVER com data Frame do Pandas:

pd.read_sql("PRAGMA table_info(silver_produtos);", conexao)


,cid,name,type,notnull,dflt_value,pk
0,0,id_produto,INTEGER,0,None,0
1,1,nome_produto,TEXT,0,None,0
2,2,categoria,TEXT,0,None,0
3,3,tipo_dado,TEXT,0,None,0
4,4,resolucao_espacial,REAL,0,None,0
5,5,sistema_coordenadas,TEXT,0,None,0
6,6,area_cobertura_km2,REAL,0,None,0
7,7,data_aquisicao,DATE,0,None,0
8,8,formato,TEXT,0,None,0
9,9,fornecedor,TEXT,0,None,0


In [ ]:
# Validação da camada Silver:
# - Tipos numéricos normalizados
# - Datas padronizadas 
# - Sem registros inválidos

